In [1]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from revenue import parse_revenue_table, common_range

In [2]:
tsla = yf.Ticker("TSLA")

In [3]:
tesla_data = tsla.history(period = "max")

In [4]:
tesla_data.reset_index(inplace=True)
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


**Data note:** the revenue tables are a frozen IBM Skills Network course snapshot (Tesla quarterly revenue to 2022-09-30, GameStop to 2020-04-30), while the prices come live from Yahoo Finance. The charts therefore only cover the dates both series have. Prices from `history()` are split- and dividend-adjusted.

In [5]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"

html_data = requests.get(url)

In [6]:
# Parse the 'Tesla Quarterly Revenue' table (found by its header, not by position) into
# numeric revenue ($US millions) and real dates; empty rows are dropped.
tesla_revenue = parse_revenue_table(html_data.content, match="Quarterly Revenue")
print(tesla_revenue.dtypes)
print(tesla_revenue.tail())

Date       datetime64[us]
Revenue           float64
dtype: object
         Date  Revenue
48 2021-09-30  13757.0
49 2021-12-31  17719.0
50 2022-03-31  18756.0
51 2022-06-30  16934.0
52 2022-09-30  21454.0


In [7]:
gme = yf.Ticker("GME")

In [8]:
gme_data = gme.history(period = "max")

In [9]:
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691667,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683250,1.687458,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578048,1.607505,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662209,1.603296,1.662209,6892800,0.0,0.0


In [10]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html"

html_data = requests.get(url)

In [11]:
# Parse the 'GameStop Quarterly Revenue' table (found by its header, not by position) into
# numeric revenue ($US millions) and real dates; empty rows are dropped.
gme_revenue = parse_revenue_table(html_data.content, match="Quarterly Revenue")
print(gme_revenue.dtypes)
print(gme_revenue.tail())

Date       datetime64[us]
Revenue           float64
dtype: object
         Date  Revenue
57 2019-04-30   1548.0
58 2019-07-31   1286.0
59 2019-10-31   1439.0
60 2020-01-31   2194.0
61 2020-04-30   1021.0


In [12]:
def make_graph(stock_data, revenue_data, stock):
    """Price and revenue over the date range both series cover. Revenue is numeric, so the axis is to scale."""
    start, end = common_range(stock_data, revenue_data)
    dates = pd.to_datetime(stock_data.Date).dt.tz_localize(None)
    stock_data_specific = stock_data[(dates >= start) & (dates <= end)]
    revenue_data_specific = revenue_data[(revenue_data.Date >= start) & (revenue_data.Date <= end)]

    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.3,
                        subplot_titles=("Historical Share Price (split/dividend-adjusted)",
                                        f"Historical Quarterly Revenue (IBM course snapshot, last quarter {revenue_data.Date.max():%Y-%m-%d})"))
    fig.add_trace(go.Scatter(x=dates[stock_data_specific.index], y=stock_data_specific.Close.astype("float"),
                             name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue_data_specific.Date, y=revenue_data_specific.Revenue,
                             name="Revenue"), row=2, col=1)

    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Adjusted close ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)   # numeric axis, no reversal needed
    fig.update_layout(showlegend=False, height=900, title=f"{stock} ({start:%Y-%m-%d} – {end:%Y-%m-%d})",
                      xaxis_rangeslider_visible=True)
    fig.show()
    return fig


In [13]:
make_graph(tesla_data, tesla_revenue, "Tesla")

In [14]:
make_graph(gme_data, gme_revenue, "GameStop")